# 06 邏輯斯迴歸：同時調整多個危險因子

Ch05 的分層分析一次只能控制一個干擾因子。這堂課用邏輯斯迴歸同時調整所有因子，
回答：**控制年齡、共病、功能狀態後，淋浴使用還是獨立危險因子嗎？**

流程：**資料準備 → Crude OR 彙整 → 多變項模型 → Table 2 → Crude vs Adjusted 比較 → 模型診斷**

In [ ]:
# Google Colab setup -- 若在本機執行可跳過此 cell
import sys
import os
if 'google.colab' in sys.modules:
    !git clone https://github.com/ancientsky/python4epi.git /content/python4epi 2>/dev/null || true
    os.chdir('/content/python4epi')
    !pip install -q -e .

In [ ]:
# --- Step 1: 資料準備 ---
import pathlib

import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

# -- CJK font setup (避免中文標籤顯示為方框) --
# 掃描系統字型目錄，顯式註冊 CJK 字型（比依賴快取更可靠）
for _font_dir in map(pathlib.Path, ["/usr/share/fonts", "/usr/local/share/fonts"]):
    if _font_dir.exists():
        for _fp in sorted(_font_dir.rglob("*")):
            if _fp.suffix.lower() in {".ttf", ".ttc", ".otf"} and (
                "CJK" in _fp.name or "WenQuanYi" in _fp.name or "wqy" in _fp.name
            ):
                try:
                    fm.fontManager.addfont(str(_fp))
                except Exception:
                    pass

plt.rcParams["font.sans-serif"] = [
    "Noto Sans CJK TC", "Noto Sans CJK SC", "Noto Sans CJK JP",
    "Noto Sans TC", "Microsoft JhengHei",
    "WenQuanYi Zen Hei", "SimHei", "Arial Unicode MS",
    "Heiti TC", "DejaVu Sans",
]
plt.rcParams["axes.unicode_minus"] = False
plt.style.use("ggplot")
plt.rcParams["figure.dpi"] = 150

df = pd.read_csv("data/synthetic/legionella_outbreak.csv")
df["infected"] = (df["clinical_severity"] != "not_ill").astype(int)

# 功能狀態轉數值（bedridden=0, wheelchair=1, ambulatory=2）
fs_map = {"bedridden": 0, "wheelchair": 1, "ambulatory": 2}
# smoking_history 是三分類（never/former/current），轉為二分類
df["ever_smoker"] = (df["smoking_history"] != "never").astype(int)

df["functional_score"] = df["functional_status"].map(fs_map)

print(f"全體：{len(df)} 人，感染：{df['infected'].sum()} 人")
print(f"侵襲率：{df['infected'].mean():.1%}")

## OR 與 RR 的關係

邏輯斯迴歸的輸出是 **Odds Ratio (OR)**，不是 Risk Ratio (RR)。

- 疾病盛行率低（<10%）→ OR ≈ RR
- 本資料集侵襲率 ~43% → OR 會比 RR 偏大，解讀時要注意
- 但 OR 的**方向**（>1 或 <1）和**統計顯著性**仍然可靠

In [ ]:
# --- Step 2: 單變項 Crude OR 彙整 ---
import warnings

factors = [
    "shower_use", "hydrotherapy_use", "ever_smoker",
    "comorbidity_chf", "comorbidity_dm", "comorbidity_cancer",
    "comorbidity_copd", "immunosuppressed",
    "age", "functional_score",
]

crude_results = []
for var in factors:
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        model = smf.logit(f"infected ~ {var}", data=df).fit(disp=0, method="lbfgs")

    if not model.mle_retvals["converged"]:
        print(f"⚠ {var}: 模型未收斂（可能存在準完美分離），跳過")
        continue

    coef = model.params[var]
    ci = model.conf_int().loc[var]
    crude_results.append({
        "variable": var,
        "crude_OR": round(np.exp(coef), 3),
        "95% CI": f"{np.exp(ci[0]):.3f}–{np.exp(ci[1]):.3f}",
        "p-value": round(model.pvalues[var], 4),
    })

crude_df = pd.DataFrame(crude_results)
print(crude_df.to_string(index=False))

In [ ]:
# --- Step 3: 多變項 Adjusted OR ---
formula = (
    "infected ~ shower_use + hydrotherapy_use + age + "
    "comorbidity_chf + comorbidity_dm + comorbidity_cancer + "
    "comorbidity_copd + immunosuppressed + functional_score + "
    "C(floor)"
)

with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    model_full = smf.logit(formula, data=df).fit(disp=0, method="lbfgs")
print(model_full.summary2())

In [ ]:
# --- Step 4: Adjusted OR 表格（Table 2 格式）---
adj_results = []
for var in model_full.params.index:
    if var == "Intercept":
        continue
    coef = model_full.params[var]
    ci = model_full.conf_int().loc[var]
    adj_results.append({
        "variable": var,
        "adjusted_OR": round(np.exp(coef), 3),
        "95% CI": f"{np.exp(ci[0]):.3f}\u2013{np.exp(ci[1]):.3f}",
        "p-value": round(model_full.pvalues[var], 4),
    })

adj_df = pd.DataFrame(adj_results)
print("=== Adjusted OR（Table 2）===")
print(adj_df.to_string(index=False))

In [ ]:
# --- Step 5: Crude vs. Adjusted OR 比較 ---
key_vars = ["shower_use", "hydrotherapy_use", "age",
            "comorbidity_chf", "immunosuppressed", "functional_score"]

comparison = []
for var in key_vars:
    crude_row = crude_df[crude_df["variable"] == var]
    adj_row = adj_df[adj_df["variable"] == var]
    if len(crude_row) == 0 or len(adj_row) == 0:
        continue
    c_or = crude_row.iloc[0]["crude_OR"]
    a_or = adj_row.iloc[0]["adjusted_OR"]
    comparison.append({
        "variable": var,
        "crude_OR": c_or,
        "adjusted_OR": a_or,
        "change": f"{((a_or - c_or) / c_or * 100):+.1f}%",
    })

comp_df = pd.DataFrame(comparison)
print("=== Crude OR vs. Adjusted OR ===")
print(comp_df.to_string(index=False))
print("\n\u2192 \u8b8a\u5316\u5e45\u5ea6\u5927\u7684\u8b8a\u9805\u4ee3\u8868\u53d7\u5230\u4ea4\u7d61\u5f71\u97ff\u8f03\u5927")

In [ ]:
# --- 視覺化：Adjusted OR 森林圖 ---
plot_df = adj_df[~adj_df["variable"].str.startswith("C(")].copy()
plot_df["ci_lo"] = plot_df["95% CI"].str.split("\u2013").str[0].astype(float)
plot_df["ci_hi"] = plot_df["95% CI"].str.split("\u2013").str[1].astype(float)

fig, ax = plt.subplots(figsize=(8, 5))
y_pos = range(len(plot_df))

ax.errorbar(
    plot_df["adjusted_OR"], y_pos,
    xerr=[plot_df["adjusted_OR"] - plot_df["ci_lo"],
          plot_df["ci_hi"] - plot_df["adjusted_OR"]],
    fmt="o", color="#2c7fb8", capsize=4, markersize=8,
)
ax.axvline(x=1, color="gray", linestyle="--", alpha=0.5)
ax.set_yticks(list(y_pos))
ax.set_yticklabels(plot_df["variable"])
ax.set_xlabel("Adjusted Odds Ratio")
ax.set_title("\u591a\u8b8a\u9805\u908f\u8f2f\u65af\u8ff4\u6b78 \u2014 Adjusted OR \u68ee\u6797\u5716")
plt.tight_layout()
plt.show()

In [ ]:
# --- Step 6: 模型診斷（AIC 比較）---
formula_reduced = (
    "infected ~ shower_use + hydrotherapy_use + age + "
    "immunosuppressed + functional_score"
)
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    model_reduced = smf.logit(formula_reduced, data=df).fit(disp=0, method="lbfgs")

print("=== 模型比較 ===")
print(f"  完整模型（10 變項）AIC = {model_full.aic:.1f}")
print(f"  精簡模型（5 變項） AIC = {model_reduced.aic:.1f}")
print(f"  \u2192 AIC 較小的模型在解釋力與複雜度間取得較好平衡")
print(f"\n  完整模型 Pseudo R\u00b2 = {model_full.prsquared:.4f}")
print(f"  精簡模型 Pseudo R\u00b2 = {model_reduced.prsquared:.4f}")

## 小結

| 步驟 | 學到的技能 |
|------|------------|
| Crude OR | `smf.logit()` 單變項迴歸 + 迴圈 |
| Adjusted OR | 多變項公式、`C()` 處理類別變項 |
| Table 2 | 從模型物件提取 OR/CI/p |
| Crude vs Adjusted | 比較干擾效應的方向和幅度 |
| 森林圖 | `errorbar` 視覺化 OR |
| 模型診斷 | AIC 比較、Pseudo R² |

**結論**：如果淋浴使用在調整後 OR 仍顯著大於 1，可以更有信心地說它是獨立危險因子。

下一章（Ch07），我們轉向時間序列預測——主管想知道「下週還會有多少新個案？」